In [ ]:
!pip install -q tensorflow scikit-learn pandas tqdm opencv-python mediapipe

In [ ]:
import time
from dataclasses import dataclass
from pathlib import Path

import cv2
import joblib
import mediapipe as mp
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tqdm import tqdm

In [ ]:
DATA_DIR = Path('/kaggle/input/asl-alphabet')
MODELS_DIR = Path('/kaggle/working/models')
RESULTS_DIR = Path('/kaggle/working/results')
CACHE_DIR = RESULTS_DIR / 'landmarks_cache'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
@dataclass
class DataConfig:
    data_dir: Path
    val_split: float = 0.15
    test_split: float = 0.15
    seed: int = 42


class ASLDataLoader:
    TRAIN_SUBDIR = 'asl_alphabet_train'

    def __init__(self, config):
        self.config = config
        self.train_dir = Path(config.data_dir) / self.TRAIN_SUBDIR
        self._df = None
        self.classes = None
        self.class_to_idx = None

    @property
    def num_classes(self):
        return len(self.classes) if self.classes else 0

    @property
    def df(self):
        if self._df is None:
            self._df = self._scan_dataset()
        return self._df

    def _scan_dataset(self):
        class_dirs = sorted(d for d in self.train_dir.iterdir() if d.is_dir())
        self.classes = [d.name for d in class_dirs]
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        records = [
            {'filepath': str(img), 'label': cls_dir.name, 'class_idx': self.class_to_idx[cls_dir.name]}
            for cls_dir in class_dirs
            for img in cls_dir.glob('*.jpg')
        ]
        return pd.DataFrame(records)

    def split(self):
        val_test_size = self.config.val_split + self.config.test_split
        relative_test = self.config.test_split / val_test_size
        train_df, temp_df = train_test_split(
            self.df, test_size=val_test_size, stratify=self.df['label'], random_state=self.config.seed,
        )
        val_df, test_df = train_test_split(
            temp_df, test_size=relative_test, stratify=temp_df['label'], random_state=self.config.seed,
        )
        return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

In [ ]:
NUM_LANDMARKS = 21
FEATURE_DIM = NUM_LANDMARKS * 3


@dataclass
class MediaPipeConfig:
    num_classes: int = 29
    dropout_rate: float = 0.3
    l2_lambda: float = 1e-4
    learning_rate: float = 1e-3
    min_detection_confidence: float = 0.5
    n_estimators: int = 200


class LandmarkExtractor:
    def __init__(self, config):
        self.config = config
        self._hands = mp.solutions.hands.Hands(
            static_image_mode=True,
            max_num_hands=1,
            min_detection_confidence=config.min_detection_confidence,
        )

    def extract_from_image(self, image_bgr):
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        result = self._hands.process(image_rgb)
        if not result.multi_hand_landmarks:
            return None
        landmarks = result.multi_hand_landmarks[0].landmark
        features = np.array([[lm.x, lm.y, lm.z] for lm in landmarks], dtype=np.float32)
        wrist = features[0]
        normalized = features - wrist
        scale = np.max(np.abs(normalized)) + 1e-6
        return (normalized / scale).flatten()

    def process_dataframe(self, df):
        features, labels, valid_idx = [], [], []
        for idx, row in tqdm(df.iterrows(), total=len(df), desc='Extracting landmarks'):
            img = cv2.imread(row['filepath'])
            if img is None:
                continue
            feat = self.extract_from_image(img)
            if feat is None:
                continue
            features.append(feat)
            labels.append(row['class_idx'])
            valid_idx.append(idx)
        return np.array(features, dtype=np.float32), np.array(labels, dtype=np.int32)

    def close(self):
        self._hands.close()


def build_landmark_classifier(config):
    inputs = Input(shape=(FEATURE_DIM,))
    x = Dense(256, activation='relu', kernel_regularizer=l2(config.l2_lambda))(inputs)
    x = BatchNormalization()(x)
    x = Dropout(config.dropout_rate)(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(config.l2_lambda))(x)
    x = BatchNormalization()(x)
    x = Dropout(config.dropout_rate)(x)
    x = Dense(64, activation='relu', kernel_regularizer=l2(config.l2_lambda))(x)
    x = Dropout(config.dropout_rate)(x)
    outputs = Dense(config.num_classes, activation='softmax')(x)
    model = Model(inputs, outputs, name='landmark_classifier')
    model.compile(optimizer=Adam(config.learning_rate), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
loader = ASLDataLoader(DataConfig(data_dir=DATA_DIR))
train_df, val_df, test_df = loader.split()
print(f'Classes: {loader.num_classes}')
print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')

In [ ]:
mediapipe_config = MediaPipeConfig(num_classes=loader.num_classes)
extractor = LandmarkExtractor(mediapipe_config)

def load_or_extract(name, df):
    X_path = CACHE_DIR / f'X_{name}.npy'
    y_path = CACHE_DIR / f'y_{name}.npy'
    if X_path.exists() and y_path.exists():
        return np.load(X_path), np.load(y_path)
    X, y = extractor.process_dataframe(df)
    np.save(X_path, X)
    np.save(y_path, y)
    return X, y

X_train, y_train = load_or_extract('train', train_df)
X_val,   y_val   = load_or_extract('val',   val_df)
X_test,  y_test  = load_or_extract('test',  test_df)

extractor.close()
print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

In [ ]:
dense_model = build_landmark_classifier(mediapipe_config)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
    ModelCheckpoint(str(MODELS_DIR / 'mediapipe_dense_best.h5'), monitor='val_accuracy', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7),
]

t0 = time.time()
history = dense_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=64,
    callbacks=callbacks,
)
dense_time = time.time() - t0
print(f'Dense train time: {dense_time:.1f}s')

pd.DataFrame(history.history).to_csv(RESULTS_DIR / 'mediapipe_dense_history.csv', index=False)

In [ ]:
rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

t0 = time.time()
rf.fit(X_train, y_train)
rf_time = time.time() - t0
print(f'RF train time: {rf_time:.1f}s')

joblib.dump(rf, MODELS_DIR / 'random_forest.pkl')

In [ ]:
best_dense = load_model(str(MODELS_DIR / 'mediapipe_dense_best.h5'))
probs_dense = best_dense.predict(X_test, verbose=0)
y_pred_dense = np.argmax(probs_dense, axis=1)

print('=== Dense Network ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_dense):.4f}')
print(classification_report(y_test, y_pred_dense, target_names=loader.classes))

probs_rf = rf.predict_proba(X_test)
y_pred_rf = np.argmax(probs_rf, axis=1)

print('=== Random Forest ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}')
print(classification_report(y_test, y_pred_rf, target_names=loader.classes))